In [ ]:
!pip install wikipedia python-docx requests

In [ ]:
import requests
import wikipedia
from docx import Document
import sys

class FocusShieldAI:
    def __init__(self):
        self.chat_history = []
        self.api_url = "http://127.0.0.1:1234/v1/chat/completions"
        self.model_id = self._get_active_model()
        print(f"FocusShield AI Initialized with model: {self.model_id}")

    def _get_active_model(self):
        """Automatically fetches the ID of the loaded model."""
        try:
            response = requests.get("http://127.0.0.1:1234/v1/models")
            data = response.json().get("data", [])
            if data:
                return data[0]["id"]
            return "unknown"
        except:
            return "error_no_model_loaded"

    def ask_ai(self, prompt):
        if self.model_id == "error_no_model_loaded":
            return "Error: No model loaded in LM Studio. Please load a model."
            
        payload = {
            "model": self.model_id,
            "messages": self.chat_history + [{"role": "user", "content": prompt}],
            "temperature": 0.7
        }
        try:
            response = requests.post(self.api_url, json=payload, timeout=30)
            if response.status_code != 200:
                return f"API Error {response.status_code}: {response.json().get('error', {}).get('message', 'Unknown Error')}"
            
            reply = response.json()['choices'][0]['message']['content']
            self.chat_history.append({"role": "user", "content": prompt})
            self.chat_history.append({"role": "assistant", "content": reply})
            return reply
        except Exception as e:
            return f"Connection Error: {e}"

    def safe_math(self, expr):
        try:
            if not all(c in "0123456789+-*/(). " for c in expr): return "Invalid math."
            return eval(expr, {"__builtins__": None}, {})
        except: return "Math error."

    def create_doc(self, topic):
        content = self.ask_ai(f"Write a detailed document about: {topic}")
        doc = Document()
        doc.add_heading(topic, 0)
        doc.add_paragraph(content)
        filename = f"{topic.replace(' ', '_')[:10]}.docx"
        doc.save(filename)
        return f"Document saved: {filename}"

# --- Execution ---
if __name__ == "__main__":
    bot = FocusShieldAI()
    print("FocusShield AI Ready. (Type 'exit' to quit)")
    
    while True:
        user_input = input("\nUser: ").strip()
        if user_input.lower() == 'exit': break
        
        if all(c in "0123456789+-*/(). " for c in user_input) and any(c in "+-*/" for c in user_input):
            print(f"AI (Math): {bot.safe_math(user_input)}")
        elif user_input.lower().startswith(("what is", "define")):
            print(f"AI (Wiki): {wikipedia.summary(user_input, sentences=2)}")
        elif "document" in user_input.lower():
            print(bot.create_doc(user_input.replace("document", "")))
        else:
            print(f"AI: {bot.ask_ai(user_input)}")

FocusShield AI Initialized with model: llama-3.2-1b-instruct
FocusShield AI Ready. (Type 'exit' to quit)
AI: Hello! How can I assist you today?
AI (Math): 2
Document saved: Write_a_su.docx
